In [1]:
import spacy
from scispacy.linking import EntityLinker
from scispacy.abbreviation import AbbreviationDetector
import os
import json
import re
from periodictable import elements
from pathlib import Path
from pypdf import PdfReader as pr
#!pip install nltk
from nltk.tokenize import sent_tokenize
import pubchempy as pcp

import warnings
warnings.filterwarnings("ignore")

In [2]:
#os.chdir('..\\dissertation DLC content')
#linker = Ontology('OntoBiotope_BioNLP-OST-2019.obo')

path_to_scispacy_model = os.path.normpath('C:\\Users\\jace\\Documents\\assignments\\dissertation DLC content\\en_ner_bionlp13cg_md-0.5.4\\en_ner_bionlp13cg_md\\en_ner_bionlp13cg_md-0.5.4')
path_to_microbial_model = os.path.normpath('C:\\Users\\jace\\Documents\\assignments\\dissertation DLC content\\custom_microbial_ner')

scispacy_model = spacy.load(path_to_scispacy_model)
microbial_model = spacy.load(path_to_microbial_model)

""" print('Reading .obo file')
onto = obonet.read_obo(obo_path)

kb = KnowledgeBase(get_entities_from_obonet(onto))

linker = scispacy_model.add_pipe('scispacy_linker')
linker.set_kb(kb) """

print('Adding entity linker')
linker = scispacy_model.add_pipe('scispacy_linker', config={'resolve_abbreviations': True, 'linker_name': 'mesh'})

microbial_model.add_pipe('abbreviation_detector', last=True, validate=False)

Adding entity linker


# <font size=4>hidden</font>

In [3]:
%%script False

print(scispacy_model.pipe_names)

#linker = pyobo.from_obo_path('..\\dissertation DLC content\\chebi_lite.obo', version='1.2')
kb = KnowledgeBase('..\\dissertation DLC content\\chebi_lite.json')

linker = scispacy_model.add_pipe('entity_linker', after='ner')
linker.set_kb(kb)

onto = pyobo.fr('chebi_core')

#update this here to try and download a more relevant knowledge base
""" !python -m spacy_entity_linker "download_knowledge_base"

from spacy.kb import InMemoryLookupKB

def create_kb(vocab):
    kb = InMemoryLookupKB(vocab, entity_vector_length=128)
    return kb

linker = scispacy_model.add_pipe('entityLinker', after='ner') """

Couldn't find program: 'False'


# <font size=4>not hidden</font>

In [ ]:
with open('list_files\\utilizations_list.json', 'r', encoding='utf-8') as f:
    utilizations = list(json.load(f))

basic_utilizations = ['fermentation', 'hydrolyze', 'hydrolyse', 'reduce', 'reduces', 'reduced', 'produce', 'production', 'produces', 'consume', 'consumes', 'consumption', 'consuming', 'producing', 'grow', 'growth', 'oxidise', 'oxidize', 'consumed', 'produced', 'convert', 'converts', 'converting', 'converted', 'conversion']

print('|'.join(basic_utilizations))

util_mappings = {
    'production': 'PRODUCES',
    'produce': 'PRODUCES',
    'produces': 'PRODUCES'
}
    
custom_labels = ['MICROBE_NAME', 'SIMPLE_CHEMICAL', 'CHEMICAL']

not_chemicals = ['flavour', 'flavor', 'flavours', 'flavors', 'carbohydrate', 'carbohydrates', 'vitamin', 'vitamins', 'aroma', 'aromas', 'flavonoid', 'flavonoids'] + [el.name for el in elements]

print(len(not_chemicals))

microbes_known_mappings = {
    'S. cerevisiae': 'Saccharomyces cerevisiae'
}

with open('list_files\\genus_names.json', 'r', encoding='utf-8') as f:
    genera_unfiltered = list(json.load(f))
    
genus_names = []

for gen in genera_unfiltered:
    if 'Candidatus' in gen:
        gen_split = gen.split(' ')
        genus_names.append(gen_split[1])
    else:
        genus_names.append(gen)
        
with open('list_files\\microbes_genus_species.json', 'r', encoding='utf-8') as f:
    microbes_genus_species = list(json.load(f))
    
g_species = []
genus_species = []
for gs in microbes_genus_species:
    genus, species = gs.split(' ')
    g_species.append(str(genus[0] + '. ' + species))
    genus_species.append((genus, species))
    
print(len(g_species), len(set(g_species)))
    
with open('list_files\\species_names.json', 'r', encoding='utf-8') as f:
    species_names = list(set(json.load(f)))

fermentation|hydrolyze|hydrolyse|reduce|reduces|reduced|produce|production|produces|consume|consumes|consumption|consuming|producing|grow|growth|oxidise|oxidize|consumed|produced|convert|converts|converting|converted|conversion
130
25174 21882


In [ ]:
for ind, file in enumerate(os.listdir("..\\dissertation DLC content\\fermentation_papers_preprocessed")):
    microbe_shortform_mappings = {}
    print(ind)
    #keep track of longterm names of all microbes seen so far
    microbes_seen = set()
    filename = os.fsdecode(file)
    if filename.endswith('json'):
        with open(f"..\\dissertation DLC content\\fermentation_papers_preprocessed\\{filename}", 'r') as f:
            text = f.read()
    elif filename.endswith('pdf'):
        try:
            print('pdf!')
            path = Path(os.getcwd() + "\\fermentation_papers\\" + filename)
            reader = pr(path)
            text = ""
            for page in reader.pages:
                text = text + page.extract_text()
        except:
            continue
    else:
        continue
    #do a first pass of the paper to identify the microbes
    all_paper_microbial = microbial_model(text)
    all_mic_ents = [(mic.label_, mic.text, mic.start_char, mic.end_char) for mic in all_paper_microbial.ents if mic.label_ == 'MICROBE_NAME']
    all_mic_shapes_raw = [mic.shape_ for mic in all_paper_microbial]
    mic_shapes = [(name, ' '.join([msr[:4] for msr in all_mic_shapes_raw[s:e]]), sc, ec) for s, e, name, sc, ec in [(m.start, m.end, m.text, m.start_char, m.end_char) for m in all_paper_microbial.ents if m.label_ == 'MICROBE_NAME']]
    for name, shape, s, e in mic_shapes:
        if shape == 'Xxxx xxxx':
            name_split = name.split(' ')
            shortform_name = name_split[0][0] + '. ' + name_split[1]
            if not microbe_shortform_mappings.get(shortform_name):
                microbe_shortform_mappings[shortform_name] = name
        elif shape == 'X. xxxxx' or shape == 'X. xxxx' or shape == 'Xx. xxxxx':
            if not microbe_shortform_mappings.get(name):
                #add that entry in to indicate we have seen the shortform before the longform, in case the shortform doesnt get seen again, sometimes the microbe names are weird
                microbe_shortform_mappings[name] = None
    
    sentences = sent_tokenize(text)
    for sent in sentences:
        mic_ents_raw = microbial_model(sent)
        mic_ents = [(mic.label_, mic.text, mic.start_char, mic.end_char) for mic in mic_ents_raw.ents if mic.label_ == 'MICROBE_NAME']
        mic_shapes_raw = [mic.shape_ for mic in mic_ents_raw]
        mic_shapes = [(name, ' '.join([msr[:4] for msr in mic_shapes_raw[s:e]]), sc, ec) for s, e, name, sc, ec in [(m.start, m.end, m.text, m.start_char, m.end_char) for m in mic_ents_raw.ents if m.label_ == 'MICROBE_NAME']]
        mic_ents_resolved = []
        for name, shape, s, e in mic_shapes:
            if shape == 'X. xxxxx' or shape == 'X. xxxx' or shape == 'Xx. xxxxx':
                #check the map to see if theres anything in there
                resolved_name = microbe_shortform_mappings.get(name)
                #add the resolved name to the array
                if resolved_name:
                    mic_ents_resolved.append(('MICROBE_NAME', resolved_name, s, e))
                else:
                    #try with the known mappings (obviously not gonna do them all, just the most common ones)
                    known_name = microbes_known_mappings.get(name)
                    if known_name:
                        mic_ents_resolved.append(('MICROBE_NAME', known_name, s, e))
                    else:
                        #perhaps match against the list of genera and species to see which is the most probable? like start with the first letter and figure shit out from there
                        #like check the species name and check it against the full names, and see if theres any genera which match up
                        #but what if there are multiple candidates?
                        
                        genus, species = name.split(' ') #genus is first letter, species is 'xxxx' bit after
                        g_species = genus[0] + '. ' + species
                        matched_species = [gen + ' ' + spec for gen, spec in genus_species if spec == species]
                        print(matched_species)
                        
                        if len(matched_species) == 1:
                            mic_ents_resolved.append(('MICROBE_NAME', matched_species[0], s, e))
                        
                        #once we have the microbe name we then must check it against ncbi to normalize it, in case thats an old name of a microbe etc. such as gbif
                        #TODO: use gbif to normalise the microbe names
                        
            elif shape == 'Xxxx xxxx':
                microbes_seen.add(name)
                name_split = name.split(' ')
                shortform_name = name_split[0][0] + '. ' + name_split[1]
                #then add it to 'resolved'
                mic_ents_resolved.append(('MICROBE_NAME', name, s, e))
        if mic_ents != mic_ents_resolved:
            print("microbial mismatch")
            print(sent[:300])
            print(mic_ents)
            print(mic_ents_resolved)
        sci_ents_raw = scispacy_model(sent)
        
        chem_ents = []
        
        for sci in sci_ents_raw.ents:
            if 'CHEMICAL' in sci.label_ and sci.text.lower() not in not_chemicals + [text.lower() for l, text, s, e in mic_ents]:
                comps = pcp.get_compounds(sci.text, 'name')
                if comps:
                    chem_ents.append(('CHEMICAL', comps[0].synonyms[0], sci.start_char, sci.end_char))
                else:
                    chem_ents.append(('CHEMICAL', sci.text, sci.start_char, sci.end_char))

        #simple chemicals should NOT be less than 3 characters long
        utils_found = re.finditer('|'.join(utilizations), str(sent).lower())
        utilization_ents = [('UTILIZATION', sent[util.start(0):util.end(0)].lower(), util.start(0), util.end(0)) for util in utils_found]
        ents_and_labels = [(label, text, start, end) for label, text, start, end in mic_ents + chem_ents if label in custom_labels and text not in not_chemicals and len(text)>=3] + utilization_ents
        ents_and_labels = sorted(ents_and_labels, key= lambda tup: tup[2])
        """ if ents_and_labels:
            print(sent[:300])
            print(ents_and_labels)
            print(chem_ents)
            print(mic_ents_raw._.abbreviations) """
            
        """ sci_lents = linker(sci_ents)
        linker_labels = []
        for ent in sci_ents.ents:
            #print([linker.kb.cui_to_entity[lent[0]] for lent in ent._.kb_ents])
            if 'CHEMICAL' in ent.label_ and ent.text.lower() not in not_chemicals + [ent.text.lower() for l, text, s, e in mic_ents]:
                if ent._.kb_ents:
                    #print(ent._.kb_ents[0])
                    label = linker.kb.cui_to_entity[ent._.kb_ents[0][0]]
                    linker_labels.append(label.canonical_name)
                else:
                    linker_labels.append(ent.text) 
                    
        linked_sci_ents = [lent for lent in sci_lents.ents if 'CHEMICAL' in lent.label_] """
    print()

0
microbial mismatch
Stanbury A. Whitaker S.J.
[('MICROBE_NAME', 'A. Whitaker', 9, 20)]
[]

1
microbial mismatch
The cultivation of wine Leuconostoc spp.
[('MICROBE_NAME', 'Leuconostoc spp', 24, 39)]
[]
microbial mismatch
Leuconostoc A simple identification procedure is applicable to Leuconostoc spp.
[('MICROBE_NAME', 'Leuconostoc spp', 63, 78)]
[]
microbial mismatch
is isolated from wine, it is automatically classified as L. oenos, as only strains of this species will grow in the presence of 10% ethanol at pH values less than 4.2.
[('MICROBE_NAME', 'L. oenos', 57, 65)]
[('MICROBE_NAME', 'Leuconostoc oenos', 57, 65)]
microbial mismatch
Growth, compared to that of the non-acidophilic Leuconostoc spp., is slow and takes 5–7 days at 22°C.
[('MICROBE_NAME', 'Leuconostoc spp', 48, 63)]
[]


KeyError: 'P. damnosus'